# ML Model Preparation — EDA, Feature Selection & Model-Ready DataFrames
---
**Input**: Feature-engineered parquet from Notebook 03
**Target**: `is_aml` (binary: 0=Clean, 1=AML)
**Output**: Train/test splits, selected features, correlation analysis, baseline model

**Pipeline**: Correlation → Feature Importance → Multicollinearity Removal → Train/Test Split → Baseline Model


## 1 — Environment Setup


In [2]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
from datetime import datetime

# Viz
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 6)

OUTPUT_DIR = "ml_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready")



Environment ready


## 2 — Load Feature-Engineered Data


In [3]:
INPUT_FILE = "../outputs_updated/stg_transactions_features_V2.parquet"

if not os.path.exists(INPUT_FILE):
    # Try alternate paths
    for alt in ["stg_transactions_features_V2.parquet",
                "../aml_features_output/stg_transactions_features_V2.parquet",
                "../stg_transactions_features_V2.parquet"]:
        if os.path.exists(alt):
            INPUT_FILE = alt
            break

df = pd.read_parquet(INPUT_FILE)
print(f"Loaded: {INPUT_FILE}")
print(f"  {len(df):,} rows × {len(df.columns)} columns")
print(f"  is_aml distribution: 0={( df['is_aml']==0).sum():,}  1={(df['is_aml']==1).sum():,}  ({(df['is_aml']==1).mean()*100:.1f}%)")



Loaded: ../outputs_updated/stg_transactions_features_V2.parquet
  333,875 rows × 318 columns
  is_aml distribution: 0=204,230  1=129,645  (38.8%)


## 3 — Initial EDA: Target Distribution & Data Overview


In [4]:
print("=" * 90)
print("INITIAL EDA")
print("=" * 90)

# ── 3.1: Target distribution ──
print("\n── 3.1: Target Variable (is_aml) ──")
target_counts = df["is_aml"].value_counts().sort_index()
for val, cnt in target_counts.items():
    print(f"  is_aml={val}: {cnt:>10,} ({cnt/len(df)*100:.1f}%)")
imbalance_ratio = target_counts[0] / max(target_counts[1], 1)
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1 (Clean:AML)")

# ── 3.2: Column type breakdown ──
print("\n── 3.2: Column Types ──")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
print(f"  Numeric: {len(numeric_cols)} | Object/String: {len(object_cols)} | Boolean: {len(bool_cols)}")

# ── 3.3: Missing values ──
print("\n── 3.3: Missing Values (top 20) ──")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
missing_top = missing_pct[missing_pct > 0].head(20)
if len(missing_top) > 0:
    for col, pct in missing_top.items():
        print(f"  {col:<50s} {pct:>6.2f}%")
else:
    print("  No missing values found")

# ── 3.4: Typology distribution within AML ──
print("\n── 3.4: Typology Distribution (within is_aml=1) ──")
if "aml_typology" in df.columns:
    aml_df = df[df["is_aml"] == 1]
    all_typs = {}
    for t in aml_df["aml_typology"].dropna():
        for part in str(t).split("; "):
            part = part.strip()
            if part: all_typs[part] = all_typs.get(part, 0) + 1
    for typ, cnt in sorted(all_typs.items(), key=lambda x: -x[1]):
        print(f"  {typ:<40s} {cnt:>8,} ({cnt/len(aml_df)*100:.1f}%)")

# ── 3.5: Key numeric feature statistics ──
print("\n── 3.5: Key Feature Statistics (AML vs Clean) ──")
key_features = ["transaction_amount", "rule_score", "fraud_intensity_score",
                "sender_acct_txn_count_24h", "sender_acct_outflow_amt_24h",
                "sender_acct_unique_counterparties_7d", "ip_risk_score"]

existing_keys = [f for f in key_features if f in df.columns]
print(f"\n  {'Feature':<45s} │ {'AML Mean':>10s} {'Clean Mean':>10s} {'Ratio':>7s} │ {'AML Med':>10s} {'Clean Med':>10s}")
print("  " + "─" * 100)
for feat in existing_keys:
    am = df.loc[df["is_aml"]==1, feat].mean()
    cm = df.loc[df["is_aml"]==0, feat].mean()
    amed = df.loc[df["is_aml"]==1, feat].median()
    cmed = df.loc[df["is_aml"]==0, feat].median()
    ratio = am / max(cm, 0.0001)
    print(f"  {feat:<45s} │ {am:>10.2f} {cm:>10.2f} {ratio:>6.2f}x │ {amed:>10.2f} {cmed:>10.2f}")

# ── 3.6: Plots ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Initial EDA — Target & Key Features", fontsize=14, fontweight="bold")

# Target bar
ax = axes[0, 0]
colors = ["#2ecc71", "#e74c3c"]
target_counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Target Distribution (is_aml)")
ax.set_xticklabels(["Clean (0)", "AML (1)"], rotation=0)
for i, v in enumerate(target_counts):
    ax.text(i, v + len(df)*0.01, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=9)

# FIS by AML status
ax = axes[0, 1]
if "fraud_intensity_score" in df.columns:
    df.loc[df["is_aml"]==0, "fraud_intensity_score"].hist(bins=50, alpha=0.6, ax=ax, label="Clean", color="#2ecc71", density=True)
    df.loc[df["is_aml"]==1, "fraud_intensity_score"].hist(bins=50, alpha=0.6, ax=ax, label="AML", color="#e74c3c", density=True)
    ax.set_title("FIS Distribution: Clean vs AML")
    ax.legend()

# Rule score by AML
ax = axes[1, 0]
if "rule_score" in df.columns:
    df.loc[df["is_aml"]==0, "rule_score"].hist(bins=50, alpha=0.6, ax=ax, label="Clean", color="#2ecc71", density=True)
    df.loc[df["is_aml"]==1, "rule_score"].hist(bins=50, alpha=0.6, ax=ax, label="AML", color="#e74c3c", density=True)
    ax.set_title("Rule Score Distribution: Clean vs AML")
    ax.legend()

# Alert level by AML
ax = axes[1, 1]
if "alert_level" in df.columns:
    ct = pd.crosstab(df["alert_level"], df["is_aml"], normalize="index") * 100
    ct = ct.reindex(["Critical", "High", "Medium", "Low", "None"])
    ct.plot(kind="barh", stacked=True, ax=ax, color=colors)
    ax.set_title("AML Rate by Alert Level")
    ax.set_xlabel("Percentage")
    ax.legend(["Clean", "AML"])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_initial_eda.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/01_initial_eda.png")



INITIAL EDA

── 3.1: Target Variable (is_aml) ──
  is_aml=0:    204,230 (61.2%)
  is_aml=1:    129,645 (38.8%)
  Imbalance ratio: 1.6:1 (Clean:AML)

── 3.2: Column Types ──
  Numeric: 226 | Object/String: 92 | Boolean: 0

── 3.3: Missing Values (top 20) ──
  No missing values found

── 3.4: Typology Distribution (within is_aml=1) ──
  Charity Abuse                              86,210 (66.5%)
  Funnel Account Network                     47,141 (36.4%)
  Third-Party Payment Web                    21,739 (16.8%)
  Pass-Through Transit Hub                    3,096 (2.4%)
  High-Risk Corridor Transfer                 3,047 (2.4%)
  Structuring (Smurfing)                      2,825 (2.2%)
  Rapid Multi-Hop Layering                    2,547 (2.0%)
  Underground Banking (Hawala)                1,704 (1.3%)
  Money Mule Network                          1,468 (1.1%)
  Circular Transaction Loop                   1,144 (0.9%)

── 3.5: Key Feature Statistics (AML vs Clean) ──

  Feature            

## 4 — Feature Classification: Identify ML-Ready Features


In [11]:
print("=" * 90)
print("FEATURE CLASSIFICATION")
print("=" * 90)

# ── Define exclusion lists ──
# LABELS: These are what we're predicting — MUST NOT be features
LABEL_COLS = {
    "is_aml", "is_aml_typology", "aml_typology", "typology_group_id", "aml_flag_source"
}

# IDENTIFIERS: Unique per transaction — no predictive value
ID_COLS = {
    "transaction_id", "timestamp", "datestamp", "customer_account_number",
    "customer_cif_id", "counterparty_account_number", "customer_name",
    "counterparty_name", "merchant_id", "merchant_name", "merchant_location",
    "session_id", "device_id_fingerprint", "ip_address", "pan", "aadhaar_number",
    "mobile_number", "email_id", "wallet_account_id", "beneficiary_wallet_id_vpa",
    "load_source_account_card_details", "customer_branch_ifsc_code",
    "counterparty_branch_ifsc_swift", "customer_cif_creation_date",
    "kyc_update_date", "account_wallet_opening_date", "account_wallet_inoperative_date",
    "date_of_birth", "date_of_incorporation",
    "father_spouse_name", "identification_proof_doc_no", "entity_identification_proof_doc_no",
    "cif_beneficial_owners", "name_beneficial_owners",
    "address_registered_office", "address_place_of_business",
    "address_beneficial_owners", "address_individual_customer",
    "place_of_incorporation", "browser_app_information",
    "geo_location_city_country", "escrow_account_linked",
    "gps_coordinates_lat", "gps_coordinates_lon",
    "customer_address_lat", "customer_address_lon"
}

# POST-HOC: Derived from is_aml — would be data leakage
POSTHOC_COLS = {
    "fraud_intensity_score", "fraud_intensity_score_raw", "fis_band",
    "alert_level", "rules_triggered", "rules_triggered_count",
    "predicted_aml", "predicted_typology", "typology_confidence"
}
# Also any prob_ columns
POSTHOC_COLS.update({c for c in df.columns if c.startswith("prob_")})

# INTERNAL: Working columns that shouldn't have survived export
INTERNAL_COLS = {c for c in df.columns if c.startswith("_")}

# STRING COLS: Need encoding first
STRING_FEATURES = {
    "transaction_type_dr_cr", "transaction_mode_channel_bank", "cash_flag",
    "transaction_type_ppi", "transaction_mode_channel_ppi", "transaction_status",
    "account_wallet_status", "pep_flag", "hni_flag", "minor_flag",
    "customer_type", "customer_entity_type", "account_category", "account_type",
    "customer_occupation_industry", "vkyc_flag", "wallet_kyc_category",
    "vpn_flag", "emulator_flag", "refund_chargeback_flag",
    "customer_current_risk_score", "tax_residency", "residency",
    "nationality", "citizenship", "non_face_to_face_flag",
    "merchant_category_code", "load_instrument_type", "authentication_method",
    "beneficial_owner_types", "passive_nfe", "source_of_funds",
    "source_of_funds_wallet", "currency"
}

all_exclude = LABEL_COLS | ID_COLS | POSTHOC_COLS | INTERNAL_COLS

# Numeric features (directly usable)
numeric_features = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in all_exclude]

# String features that need encoding (and exist in data)
string_features = [c for c in STRING_FEATURES if c in df.columns and c not in all_exclude]

# Rule flag columns (binary 0/1)
rule_flags = [c for c in df.columns if c.startswith("rule_") and c not in 
              {"rule_score", "rules_triggered", "rules_triggered_count"}
              and c not in all_exclude]

print(f"\n  Feature Classification:")
print(f"    Numeric features (direct):    {len(numeric_features)}")
print(f"    String features (to encode):  {len(string_features)}")
print(f"    Rule flags (binary):          {len(rule_flags)}")
print(f"    ─────────────────────────────────")
print(f"    Total candidate features:     {len(numeric_features) + len(string_features)}")
print(f"")
print(f"  Excluded:")
print(f"    Labels (target):              {len(LABEL_COLS)}")
print(f"    Identifiers:                  {len(ID_COLS)}")
print(f"    Post-hoc / leakage:           {len(POSTHOC_COLS)}")
print(f"    Internal working:             {len(INTERNAL_COLS)}")

# Store for later
feature_config = {
    "numeric": numeric_features,
    "string": string_features,
    "rule_flags": rule_flags,
    "labels": list(LABEL_COLS),
    "excluded": list(all_exclude)
}



FEATURE CLASSIFICATION

  Feature Classification:
    Numeric features (direct):    218
    String features (to encode):  34
    Rule flags (binary):          126
    ─────────────────────────────────
    Total candidate features:     252

  Excluded:
    Labels (target):              5
    Identifiers:                  46
    Post-hoc / leakage:           9
    Internal working:             0


## 5 — Encode Categorical Features


In [14]:
print("Encoding categorical features...")

df_ml = df.copy()

encoded_cols = []
encoding_maps = {}

for col in string_features:
    if col not in df_ml.columns:
        continue
    vals = df_ml[col].astype(str).str.strip().str.upper()
    vals = vals.replace({"NAN": "", "NONE": "", "": "MISSING"})
    
    categories = sorted(vals.unique())
    cat_map = {cat: i for i, cat in enumerate(categories)}
    encoded_col = f"{col}_enc"
    df_ml[encoded_col] = vals.map(cat_map).fillna(-1).astype(int)
    encoded_cols.append(encoded_col)
    encoding_maps[col] = cat_map
    
    n_unique = len(categories)
    print(f"  {col:<45s} → {encoded_col:<50s} ({n_unique} categories)")

print(f"\n  Total encoded columns: {len(encoded_cols)}")

# ── Save encoding maps for future use ──
import json

# 1. JSON format (human-readable, portable)
encoding_json = {col: {str(k): int(v) for k, v in mapping.items()} 
                 for col, mapping in encoding_maps.items()}
json_path = os.path.join(OUTPUT_DIR, "label_encoding_maps.json")
with open(json_path, "w") as f:
    json.dump(encoding_json, f, indent=2)
print(f"\n  Saved: {json_path}")

# 2. Pickle format (exact Python objects, fastest to reload)
import pickle
pkl_path = os.path.join(OUTPUT_DIR, "label_encoding_maps.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(encoding_maps, f)
print(f"  Saved: {pkl_path}")

# 3. CSV format (one file per feature, easy to inspect/edit)
enc_csv_dir = os.path.join(OUTPUT_DIR, "encoding_csvs")
os.makedirs(enc_csv_dir, exist_ok=True)
for col, mapping in encoding_maps.items():
    enc_df = pd.DataFrame([
        {"category": k, "encoded_value": v} for k, v in mapping.items()
    ]).sort_values("encoded_value")
    enc_df.to_csv(os.path.join(enc_csv_dir, f"{col}_encoding.csv"), index=False)
print(f"  Saved: {enc_csv_dir}/ ({len(encoding_maps)} CSV files)")

# Print summary
print(f"\n  Encoding Summary:")
print(f"  {'Column':<45s} {'Categories':>10s} {'Sample Mappings'}")
print(f"  {'─'*100}")
for col, mapping in encoding_maps.items():
    n = len(mapping)
    sample = list(mapping.items())[:3]
    sample_str = ", ".join(f"{k}→{v}" for k, v in sample)
    if n > 3:
        sample_str += f", ... (+{n-3} more)"
    print(f"  {col:<45s} {n:>10d}   {sample_str}")

# ── Helper function for encoding future data ──
print(f"""
  ╔══════════════════════════════════════════════════════════════╗
  ║  TO ENCODE FUTURE DATA, use this snippet:                   ║
  ╚══════════════════════════════════════════════════════════════╝

  import json
  with open("{json_path}") as f:
      saved_maps = json.load(f)
  
  for col, mapping in saved_maps.items():
      vals = df_new[col].astype(str).str.strip().str.upper()
      vals = vals.replace({{"NAN":"","NONE":"","":"MISSING"}})
      df_new[f"{{col}}_enc"] = vals.map(mapping).fillna(-1).astype(int)
      # -1 means unseen category (not in training data)
""")

# Final feature list
all_features = numeric_features + encoded_cols
for rf in rule_flags:
    if rf in df_ml.columns and rf not in all_features:
        all_features.append(rf)
all_features = list(dict.fromkeys(all_features))
print(f"  Total ML features: {len(all_features)}")

Encoding categorical features...
  minor_flag                                    → minor_flag_enc                                     (2 categories)
  merchant_category_code                        → merchant_category_code_enc                         (25 categories)
  source_of_funds                               → source_of_funds_enc                                (14 categories)
  refund_chargeback_flag                        → refund_chargeback_flag_enc                         (2 categories)
  customer_occupation_industry                  → customer_occupation_industry_enc                   (27 categories)
  vpn_flag                                      → vpn_flag_enc                                       (2 categories)
  cash_flag                                     → cash_flag_enc                                      (2 categories)
  citizenship                                   → citizenship_enc                                    (15 categories)
  vkyc_flag                        

## 6 — Correlation Analysis with Target (is_aml)


In [15]:
print("=" * 90)
print("CORRELATION ANALYSIS WITH TARGET (is_aml)")
print("=" * 90)

# ── 6.1: Point-biserial correlation (numeric vs binary target) ──
target = df_ml["is_aml"].astype(float)
correlations = {}

for feat in all_features:
    if feat not in df_ml.columns:
        continue
    vals = pd.to_numeric(df_ml[feat], errors="coerce").fillna(0)
    if vals.std() == 0:
        correlations[feat] = 0.0
        continue
    correlations[feat] = vals.corr(target)

corr_df = pd.DataFrame([
    {"feature": k, "correlation": v, "abs_correlation": abs(v)}
    for k, v in correlations.items()
]).sort_values("abs_correlation", ascending=False)

print(f"\n── 6.1: Top 30 Features by Correlation with is_aml ──")
print(f"\n  {'Rank':<5s} {'Feature':<55s} {'Correlation':>12s} {'Direction':>10s}")
print("  " + "─" * 85)
for i, (_, row) in enumerate(corr_df.head(30).iterrows(), 1):
    direction = "Positive ↑" if row["correlation"] > 0 else "Negative ↓"
    bar = "█" * int(abs(row["correlation"]) * 40)
    print(f"  {i:<5d} {row['feature']:<55s} {row['correlation']:>+11.6f} {direction:<10s} {bar}")

# ── 6.2: Bottom features (near zero correlation) ──
print(f"\n── 6.2: Bottom 20 Features (near zero correlation — candidates for removal) ──")
for i, (_, row) in enumerate(corr_df.tail(20).iterrows(), 1):
    print(f"  {i:<5d} {row['feature']:<55s} {row['correlation']:>+11.6f}")

# ── 6.3: Correlation heatmap (top 20 features + target) ──
top20_feats = corr_df.head(20)["feature"].tolist()
top20_with_target = top20_feats + ["is_aml"]
existing_top20 = [c for c in top20_with_target if c in df_ml.columns]

fig, ax = plt.subplots(figsize=(16, 14))
corr_matrix = df_ml[existing_top20].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Top 20 Features — Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_correlation_heatmap_top20.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/02_correlation_heatmap_top20.png")

# ── 6.4: Save correlation table ──
corr_df.to_csv(os.path.join(OUTPUT_DIR, "feature_correlations.csv"), index=False)
print(f"  Saved: {OUTPUT_DIR}/feature_correlations.csv ({len(corr_df)} features)")



CORRELATION ANALYSIS WITH TARGET (is_aml)

── 6.1: Top 30 Features by Correlation with is_aml ──

  Rank  Feature                                                  Correlation  Direction
  ─────────────────────────────────────────────────────────────────────────────────────
  1     sender_acct_txn_count_30d                                 -0.262509 Negative ↓ ██████████
  2     sender_acct_outflow_count_30d                             -0.258154 Negative ↓ ██████████
  3     sender_cust_outflow_count_30d                             -0.245337 Negative ↓ █████████
  4     sender_cust_txn_count_30d                                 -0.244617 Negative ↓ █████████
  5     sender_acct_inflow_count_30d                              -0.226073 Negative ↓ █████████
  6     sender_cust_inflow_count_30d                              -0.219960 Negative ↓ ████████
  7     receiver_account_outflow_30d                              -0.216513 Negative ↓ ████████
  8     receiver_acct_outflow_count_30d        

## 7 — Multicollinearity Detection & Feature Removal


In [16]:
print("=" * 90)
print("MULTICOLLINEARITY DETECTION")
print("=" * 90)

# ── 7.1: Find highly correlated feature pairs (|r| > 0.90) ──
THRESHOLD = 0.90

# Use only numeric features with non-zero variance
valid_feats = [f for f in all_features if f in df_ml.columns 
               and pd.to_numeric(df_ml[f], errors="coerce").std() > 0]

print(f"  Computing pairwise correlations for {len(valid_feats)} features...")
# Sample for speed if dataset is large
if len(df_ml) > 50000:
    sample_df = df_ml[valid_feats].sample(50000, random_state=42)
else:
    sample_df = df_ml[valid_feats]

corr_all = sample_df.corr()

# Find pairs above threshold
high_corr_pairs = []
for i in range(len(corr_all.columns)):
    for j in range(i+1, len(corr_all.columns)):
        r = corr_all.iloc[i, j]
        if abs(r) >= THRESHOLD:
            f1 = corr_all.columns[i]
            f2 = corr_all.columns[j]
            high_corr_pairs.append((f1, f2, r))

high_corr_pairs.sort(key=lambda x: -abs(x[2]))
print(f"\n── 7.1: Highly Correlated Pairs (|r| >= {THRESHOLD}) ──")
print(f"  Found: {len(high_corr_pairs)} pairs\n")

print(f"  {'Feature A':<45s} {'Feature B':<45s} {'Correlation':>12s}")
print("  " + "─" * 105)
for f1, f2, r in high_corr_pairs[:30]:
    print(f"  {f1:<45s} {f2:<45s} {r:>+11.4f}")
if len(high_corr_pairs) > 30:
    print(f"  ... and {len(high_corr_pairs) - 30} more pairs")

# ── 7.2: Greedy removal — keep feature with higher target correlation ──
print(f"\n── 7.2: Greedy Multicollinearity Removal ──")
print(f"  Strategy: For each correlated pair, remove the feature with LOWER |correlation| to is_aml")

target_corr = {row["feature"]: row["abs_correlation"] for _, row in corr_df.iterrows()}

to_remove = set()
for f1, f2, r in high_corr_pairs:
    if f1 in to_remove or f2 in to_remove:
        continue
    c1 = target_corr.get(f1, 0)
    c2 = target_corr.get(f2, 0)
    if c1 >= c2:
        to_remove.add(f2)
    else:
        to_remove.add(f1)

print(f"\n  Features to remove (lower target correlation): {len(to_remove)}")
for f in sorted(to_remove):
    print(f"    ✗ {f:<55s} (target corr: {target_corr.get(f, 0):.6f})")

# ── 7.3: Apply removal ──
features_after_multicollinearity = [f for f in all_features if f not in to_remove]
print(f"\n  Features before: {len(all_features)}")
print(f"  Removed:         {len(to_remove)}")
print(f"  Features after:  {len(features_after_multicollinearity)}")



MULTICOLLINEARITY DETECTION
  Computing pairwise correlations for 175 features...

── 7.1: Highly Correlated Pairs (|r| >= 0.9) ──
  Found: 20 pairs

  Feature A                                     Feature B                                      Correlation
  ─────────────────────────────────────────────────────────────────────────────────────────────────────────
  receiver_acct_outflow_count_30d               receiver_account_outflow_30d                      +1.0000
  ip_flag_vpn                                   vpn_flag_enc                                      +1.0000
  ip_flag_emulator                              emulator_flag_enc                                 +1.0000
  sender_balance_before_txn                     sender_balance_after_txn                          +0.9871
  receiver_balance_before_txn                   receiver_balance_after_txn                        +0.9868
  rule_ppi_kyc_expiry_limit                     rule_ppi_kyc_behavior_breach                      +0.9836

## MULTI COLLINEARITY ALONG WITH VIF SCORES

In [19]:
print("=" * 90)
print("MULTICOLLINEARITY DETECTION (Detection Only — No Removal)")
print("=" * 90)

# ── 7.1: Find highly correlated feature pairs (|r| > 0.90) ──
THRESHOLD = 0.90

valid_feats = [f for f in all_features if f in df_ml.columns 
               and pd.to_numeric(df_ml[f], errors="coerce").std() > 0]

print(f"  Computing pairwise correlations for {len(valid_feats)} features...")
if len(df_ml) > 50000:
    sample_df = df_ml[valid_feats].sample(50000, random_state=42)
else:
    sample_df = df_ml[valid_feats]

for c in sample_df.columns:
    sample_df[c] = pd.to_numeric(sample_df[c], errors="coerce").fillna(0)

corr_all = sample_df.corr()

high_corr_pairs = []
for i in range(len(corr_all.columns)):
    for j in range(i+1, len(corr_all.columns)):
        r = corr_all.iloc[i, j]
        if abs(r) >= THRESHOLD:
            f1 = corr_all.columns[i]
            f2 = corr_all.columns[j]
            high_corr_pairs.append((f1, f2, r))

high_corr_pairs.sort(key=lambda x: -abs(x[2]))
print(f"\n── 7.1: Highly Correlated Pairs (|r| >= {THRESHOLD}) ──")
print(f"  Found: {len(high_corr_pairs)} pairs\n")

target_corr = {row["feature"]: row["abs_correlation"] for _, row in corr_df.iterrows()}

print(f"  {'Feature A':<45s} {'Feature B':<45s} {'Correlation':>12s} {'Target Corr A':>14s} {'Target Corr B':>14s} {'Recommendation':>18s}")
print("  " + "─" * 155)
for f1, f2, r in high_corr_pairs[:50]:
    c1 = target_corr.get(f1, 0)
    c2 = target_corr.get(f2, 0)
    rec = f"Consider drop {f2}" if c1 >= c2 else f"Consider drop {f1}"
    print(f"  {f1:<45s} {f2:<45s} {r:>+11.4f} {c1:>13.6f} {c2:>13.6f}   {rec}")
if len(high_corr_pairs) > 50:
    print(f"  ... and {len(high_corr_pairs) - 50} more pairs")

# ── 7.2: Correlation band summary ──
print(f"\n── 7.2: Correlation Band Summary ──")
bands = [(0.99, 1.00), (0.95, 0.99), (0.90, 0.95), (0.80, 0.90), (0.70, 0.80)]
for lo, hi in bands:
    cnt = sum(1 for _, _, r in high_corr_pairs if lo <= abs(r) < hi)
    if lo == 0.99:
        cnt = sum(1 for _, _, r in high_corr_pairs if abs(r) >= 0.99)
        print(f"  |r| >= 0.99 (near duplicates):    {cnt:>5d} pairs")
    else:
        print(f"  {lo:.2f} <= |r| < {hi:.2f}:                 {cnt:>5d} pairs")

# ═══════════════════════════════════════════════════════════════
# 7.3: VIF (Variance Inflation Factor)
# ═══════════════════════════════════════════════════════════════
print(f"\n── 7.3: Variance Inflation Factor (VIF) ──")
print(f"  VIF = 1/(1-R²) where R² is from regressing each feature on all other features")
print(f"  VIF > 5: moderate multicollinearity | VIF > 10: severe | VIF > 50: critical\n")

from sklearn.linear_model import LinearRegression

# Use top features by importance to keep VIF computation tractable
# VIF on 200+ features is extremely slow, so we compute for top 50 by target correlation
top_vif_features = corr_df.head(50)["feature"].tolist()
top_vif_features = [f for f in top_vif_features if f in sample_df.columns]

print(f"  Computing VIF for top {len(top_vif_features)} features (by target correlation)...")

X_vif = sample_df[top_vif_features].copy()
for c in X_vif.columns:
    X_vif[c] = pd.to_numeric(X_vif[c], errors="coerce").fillna(0)

# Standardize to avoid numerical issues
X_vif = (X_vif - X_vif.mean()) / X_vif.std().replace(0, 1)

vif_results = []
lr = LinearRegression()

for i, feat in enumerate(top_vif_features):
    y_vif = X_vif[feat].values
    X_others = X_vif.drop(columns=[feat]).values
    
    try:
        lr.fit(X_others, y_vif)
        r_squared = lr.score(X_others, y_vif)
        vif = 1 / max(1 - r_squared, 0.0001)
    except:
        vif = float("inf")
    
    vif_results.append({
        "feature": feat,
        "vif": vif,
        "r_squared": r_squared if 'r_squared' in dir() else 0,
        "target_corr": target_corr.get(feat, 0)
    })
    
    if (i + 1) % 10 == 0:
        print(f"    Computed {i+1}/{len(top_vif_features)}...")

vif_df = pd.DataFrame(vif_results).sort_values("vif", ascending=False)

print(f"\n  {'Rank':<5s} {'Feature':<50s} {'VIF':>10s} {'R²':>8s} {'Target Corr':>12s} {'Severity':>12s}")
print("  " + "─" * 102)

for i, (_, row) in enumerate(vif_df.iterrows(), 1):
    vif = row["vif"]
    if vif >= 50:
        severity = "⚠ CRITICAL"
    elif vif >= 10:
        severity = "⚡ SEVERE"
    elif vif >= 5:
        severity = "● MODERATE"
    else:
        severity = "✓ OK"
    
    vif_display = f"{vif:>10.2f}" if vif < 10000 else f"{vif:>10.0f}"
    print(f"  {i:<5d} {row['feature']:<50s} {vif_display} {row['r_squared']:>7.4f} {row['target_corr']:>11.6f} {severity}")

# ── VIF Summary ──
critical = len(vif_df[vif_df["vif"] >= 50])
severe = len(vif_df[(vif_df["vif"] >= 10) & (vif_df["vif"] < 50)])
moderate = len(vif_df[(vif_df["vif"] >= 5) & (vif_df["vif"] < 10)])
ok = len(vif_df[vif_df["vif"] < 5])

print(f"\n  VIF Summary:")
print(f"    ✓ OK (VIF < 5):           {ok:>4d} features — no multicollinearity")
print(f"    ● Moderate (5 ≤ VIF < 10): {moderate:>4d} features — acceptable for tree models")
print(f"    ⚡ Severe (10 ≤ VIF < 50):  {severe:>4d} features — redundant information")
print(f"    ⚠ Critical (VIF ≥ 50):     {critical:>4d} features — nearly perfect linear dependence")

# Save VIF results
vif_df.to_csv(os.path.join(OUTPUT_DIR, "vif_scores.csv"), index=False)
print(f"\n  Saved: {OUTPUT_DIR}/vif_scores.csv")

# Save correlated pairs
pairs_df = pd.DataFrame(high_corr_pairs, columns=["feature_a", "feature_b", "correlation"])
pairs_df["target_corr_a"] = pairs_df["feature_a"].map(target_corr)
pairs_df["target_corr_b"] = pairs_df["feature_b"].map(target_corr)
pairs_df["recommendation"] = pairs_df.apply(
    lambda r: f"Consider drop {r['feature_b']}" if r["target_corr_a"] >= r["target_corr_b"] 
    else f"Consider drop {r['feature_a']}", axis=1)
pairs_df.to_csv(os.path.join(OUTPUT_DIR, "multicollinear_pairs.csv"), index=False)
print(f"  Saved: {OUTPUT_DIR}/multicollinear_pairs.csv ({len(pairs_df)} pairs)")

# ── Plot VIF distribution ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# VIF bar chart (top 30)
top30_vif = vif_df.head(30).sort_values("vif")
colors = ["#e74c3c" if v >= 50 else "#f39c12" if v >= 10 else "#3498db" if v >= 5 else "#2ecc71" 
          for v in top30_vif["vif"]]
axes[0].barh(top30_vif["feature"], top30_vif["vif"], color=colors)
axes[0].axvline(x=5, color="orange", linestyle="--", alpha=0.7, label="Moderate (5)")
axes[0].axvline(x=10, color="red", linestyle="--", alpha=0.7, label="Severe (10)")
axes[0].set_title("Top 30 Features by VIF Score", fontsize=12, fontweight="bold")
axes[0].set_xlabel("VIF")
axes[0].legend()

# VIF severity pie
sizes = [ok, moderate, severe, critical]
labels = [f"OK <5\n({ok})", f"Moderate 5-10\n({moderate})", f"Severe 10-50\n({severe})", f"Critical 50+\n({critical})"]
pie_colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]
axes[1].pie([s for s in sizes if s > 0], labels=[l for l, s in zip(labels, sizes) if s > 0],
            colors=[c for c, s in zip(pie_colors, sizes) if s > 0], autopct="%1.0f%%", startangle=90)
axes[1].set_title("VIF Severity Distribution", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_vif_analysis.png"), bbox_inches="tight")
plt.show()
print(f"  Saved: {OUTPUT_DIR}/05_vif_analysis.png")

# ── NO REMOVAL — pass all features through ──
features_after_multicollinearity = all_features.copy()
print(f"\n  ⓘ Detection only — ALL {len(features_after_multicollinearity)} features retained for model training")
print(f"  Note: LightGBM handles multicollinearity well. Features flagged above are")
print(f"  informational — remove only if model performance degrades or for interpretability.")

MULTICOLLINEARITY DETECTION (Detection Only — No Removal)
  Computing pairwise correlations for 175 features...

── 7.1: Highly Correlated Pairs (|r| >= 0.9) ──
  Found: 20 pairs

  Feature A                                     Feature B                                      Correlation  Target Corr A  Target Corr B     Recommendation
  ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  receiver_acct_outflow_count_30d               receiver_account_outflow_30d                      +1.0000      0.216513      0.216513   Consider drop receiver_account_outflow_30d
  ip_flag_vpn                                   vpn_flag_enc                                      +1.0000      0.004166      0.004166   Consider drop vpn_flag_enc
  ip_flag_emulator                              emulator_flag_enc                                 +1.0000      0.003499      0.003499   Consider drop emulator_flag_

## 8 — Feature Importance (Quick LightGBM Scan)


In [18]:
pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
print("=" * 90)
print("FEATURE IMPORTANCE — Quick LightGBM Scan")
print("=" * 90)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print("  LightGBM not installed. Run: pip install lightgbm")
    HAS_LGB = False

if HAS_LGB:
    X_imp = df_ml[features_after_multicollinearity].copy()
    for c in X_imp.columns:
        X_imp[c] = pd.to_numeric(X_imp[c], errors="coerce").fillna(0)
    y_imp = df_ml["is_aml"].astype(int)
    
    # Quick model (200 rounds, no CV)
    train_data = lgb.Dataset(X_imp, label=y_imp)
    params = {
        "objective": "binary", "metric": "auc",
        "learning_rate": 0.1, "num_leaves": 31,
        "max_depth": 6, "min_child_samples": 50,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "scale_pos_weight": (y_imp==0).sum() / max((y_imp==1).sum(), 1),
        "verbosity": -1, "random_state": 42, "n_jobs": -1
    }
    
    print("  Training quick LightGBM for feature importance...")
    model_imp = lgb.train(params, train_data, num_boost_round=200)
    
    importance_gain = pd.DataFrame({
        "feature": features_after_multicollinearity,
        "gain": model_imp.feature_importance(importance_type="gain"),
        "split": model_imp.feature_importance(importance_type="split")
    }).sort_values("gain", ascending=False)
    
    print(f"\n── Top 30 Features by Gain Importance ──")
    print(f"  {'Rank':<5s} {'Feature':<55s} {'Gain':>12s} {'Splits':>8s} {'Cum%':>7s}")
    print("  " + "─" * 90)
    total_gain = importance_gain["gain"].sum()
    cum_pct = 0
    for i, (_, row) in enumerate(importance_gain.head(30).iterrows(), 1):
        pct = row["gain"] / total_gain * 100
        cum_pct += pct
        bar = "█" * int(pct)
        print(f"  {i:<5d} {row['feature']:<55s} {row['gain']:>12.1f} {row['split']:>8.0f} {cum_pct:>6.1f}% {bar}")
    
    # ── Zero importance features ──
    zero_imp = importance_gain[importance_gain["gain"] == 0]
    print(f"\n  Zero importance features: {len(zero_imp)} (will be removed)")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(18, 10))
    
    top30 = importance_gain.head(30).sort_values("gain")
    axes[0].barh(top30["feature"], top30["gain"], color="#3498db")
    axes[0].set_title("Top 30 Features by Gain Importance", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Gain")
    
    top30_split = importance_gain.head(30).sort_values("split")
    axes[1].barh(top30_split["feature"], top30_split["split"], color="#e74c3c")
    axes[1].set_title("Top 30 Features by Split Count", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Split Count")
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_feature_importance.png"), bbox_inches="tight")
    plt.show()
    print(f"  Saved: {OUTPUT_DIR}/03_feature_importance.png")
    
    importance_gain.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"), index=False)
    
    # Remove zero importance
    features_final = importance_gain[importance_gain["gain"] > 0]["feature"].tolist()
    print(f"\n  Final feature count: {len(features_final)} (removed {len(zero_imp)} zero-importance)")
else:
    features_final = features_after_multicollinearity
    importance_gain = pd.DataFrame()



FEATURE IMPORTANCE — Quick LightGBM Scan
  Training quick LightGBM for feature importance...

── Top 30 Features by Gain Importance ──
  Rank  Feature                                                         Gain   Splits    Cum%
  ──────────────────────────────────────────────────────────────────────────────────────────
  1     sender_bal_ratio_after_to_current                           181802.4      272   16.3% ████████████████
  2     transaction_status_enc                                      180232.6      101   32.4% ████████████████
  3     receiver_bal_ratio_after_to_current                         141154.4      237   45.0% ████████████
  4     transaction_amount                                          118163.5      418   55.6% ██████████
  5     receiver_acct_txn_count_30d                                  54020.5      209   60.4% ████
  6     sender_acct_txn_count_30d                                    49689.9      161   64.8% ████
  7     sender_running_balance_txn_amount     

## 9 — Feature Selection Summary & Final Feature Set


In [23]:
print("=" * 90)
print("FEATURE SELECTION PIPELINE SUMMARY")
print("=" * 90)

stages = [
    ("All columns in dataset", len(df_ml.columns)),
    ("After removing labels/IDs/post-hoc", len(all_features)),
    ("After multicollinearity removal (|r|>0.90)", len(features_after_multicollinearity)),
    ("After zero-importance removal", len(features_final)),
]

print(f"\n  {'Stage':<55s} {'Features':>10s} {'Removed':>10s}")
print("  " + "─" * 80)
prev = None
for stage, count in stages:
    removed = f"-{prev - count}" if prev is not None else "—"
    print(f"  {stage:<55s} {count:>10,} {removed:>10s}")
    prev = count

# ── Feature category breakdown ──
print(f"\n  Final {len(features_final)} features by category:")
categories = {
    "Rule flags (rule_*)": [f for f in features_final if f.startswith("rule_")],
    "Sender account velocity": [f for f in features_final if f.startswith("sender_acct_")],
    "Sender customer velocity": [f for f in features_final if f.startswith("sender_cust_")],
    "Sender balance": [f for f in features_final if "sender" in f and any(k in f for k in ["balance","running","cumulative","pct_balance"])],
    "Receiver features": [f for f in features_final if f.startswith("receiver_")],
    "IP risk features": [f for f in features_final if f.startswith("ip_")],
    "Volume ratios": [f for f in features_final if "volume_balance_ratio" in f],
    "Encoded categoricals": [f for f in features_final if f.endswith("_enc")],
    "Transaction amount & core": [f for f in features_final if f in ["transaction_amount","annual_income","rule_score","professional_experience_years","credit_summation_period","debit_summation_period"]],
    "FIS & composite": [f for f in features_final if "fraud_intensity" in f or "typology_signal" in f or "convergence" in f or "temporal" in f],
    "Other": [],
}
categorized = set()
for cat, cols in categories.items():
    categorized.update(cols)
    if cols:
        print(f"    {cat:<35s} {len(cols):>4d}")
categories["Other"] = [f for f in features_final if f not in categorized]
if categories["Other"]:
    print(f"    {'Other':<35s} {len(categories['Other']):>4d}")

# Save feature list
with open(os.path.join(OUTPUT_DIR, "selected_features.txt"), "w") as f:
    for feat in features_final:
        f.write(feat + "\n")
print(f"\n  Saved: {OUTPUT_DIR}/selected_features.txt")



FEATURE SELECTION PIPELINE SUMMARY

  Stage                                                     Features    Removed
  ────────────────────────────────────────────────────────────────────────────────
  All columns in dataset                                         352          —
  After removing labels/IDs/post-hoc                             252       -100
  After multicollinearity removal (|r|>0.90)                     252         -0
  After zero-importance removal                                  117       -135

  Final 117 features by category:
    Rule flags (rule_*)                   14
    Sender account velocity               18
    Sender customer velocity              17
    Sender balance                         5
    Receiver features                     24
    IP risk features                       7
    Volume ratios                          2
    Encoded categoricals                  24
    Transaction amount & core              3
    FIS & composite                      

## 10 — Train/Test Split & Model-Ready DataFrames


In [26]:
df_ml.head()

,transaction_id,timestamp,datestamp,transaction_amount,currency,transaction_type_dr_cr,transaction_mode_channel_bank,cash_flag,transaction_type_ppi,transaction_mode_channel_ppi,...,hni_flag_enc,residency_enc,transaction_mode_channel_bank_enc,emulator_flag_enc,transaction_type_dr_cr_enc,customer_current_risk_score_enc,account_type_enc,source_of_funds_wallet_enc,account_wallet_status_enc,tax_residency_enc
0,TXNKKLW84T9BOTGGF8V,11:13:04,01-12-2025,184219.15,INR,Cr,Cheque,N,,,...,0,1,2,0,0,1,3,3,0,0
1,TXN3PGDUYV45WPRH3A0,06:29:34,03-12-2025,7296.96,INR,Dr,Internet Banking,N,,,...,0,1,5,0,1,1,3,3,0,0
2,TXN1S2KESOBQYUC8UVR,08:35:38,03-12-2025,15547.34,INR,Cr,NEFT,N,,,...,0,1,8,0,0,1,3,3,0,0
3,TXNM70APW4MUDKLEIXT,16:53:31,04-12-2025,2674.18,INR,Cr,NEFT,N,,,...,0,1,8,0,0,1,3,3,0,0
4,TXNOLOJGOJ70YFNNSNQ,12:56:11,06-12-2025,16066.31,INR,Dr,NEFT,N,,,...,0,1,8,0,1,1,3,3,0,0


In [24]:
print("=" * 90)
print("TRAIN/TEST SPLIT & MODEL-READY DATAFRAMES")
print("=" * 90)

from sklearn.model_selection import train_test_split

# ── Prepare X and y ──
X = df_ml[features_final].copy()
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

y = df_ml["is_aml"].astype(int)

# ── Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n  Train set: {len(X_train):,} rows ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Test set:  {len(X_test):,} rows ({len(X_test)/len(X)*100:.0f}%)")
print(f"  Features:  {X_train.shape[1]}")

print(f"\n  Target distribution:")
print(f"    {'Set':<10s} {'Total':>10s} {'AML (1)':>10s} {'Clean (0)':>10s} {'AML %':>7s}")
print(f"    {'─'*50}")
for name, yt in [("Train", y_train), ("Test", y_test)]:
    print(f"    {name:<10s} {len(yt):>10,} {(yt==1).sum():>10,} {(yt==0).sum():>10,} {(yt==1).mean()*100:>6.1f}%")

# ── Data quality check ──
print(f"\n  Data quality:")
print(f"    NaN in X_train: {X_train.isnull().sum().sum()}")
print(f"    NaN in X_test:  {X_test.isnull().sum().sum()}")
print(f"    Inf in X_train: {np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum()}")
print(f"    X_train dtype:  {X_train.dtypes.value_counts().to_dict()}")

# ── Save model-ready data ──
X_train.to_parquet(os.path.join(OUTPUT_DIR, "X_train.parquet"), index=False)
X_test.to_parquet(os.path.join(OUTPUT_DIR, "X_test.parquet"), index=False)
y_train.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "y_train.parquet"), index=False)
y_test.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "y_test.parquet"), index=False)

# Save full metadata
metadata = {
    "features": features_final,
    "n_features": len(features_final),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "aml_rate_train": float(y_train.mean()),
    "aml_rate_test": float(y_test.mean()),
    "imbalance_ratio": float((y_train==0).sum() / max((y_train==1).sum(), 1)),
    "encoding_maps": {k: {str(kk): int(vv) for kk, vv in v.items()} for k, v in encoding_maps.items()},
    "removed_multicollinear": list(to_remove),
    "correlation_threshold": THRESHOLD,
}

import json
with open(os.path.join(OUTPUT_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n  Saved model-ready files:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fn)
    size = os.path.getsize(fpath) / (1024*1024)
    print(f"    {fn:<45s} {size:>8.2f} MB")



TRAIN/TEST SPLIT & MODEL-READY DATAFRAMES

  Train set: 267,100 rows (80%)
  Test set:  66,775 rows (20%)
  Features:  117

  Target distribution:
    Set             Total    AML (1)  Clean (0)   AML %
    ──────────────────────────────────────────────────
    Train         267,100    103,716    163,384   38.8%
    Test           66,775     25,929     40,846   38.8%

  Data quality:
    NaN in X_train: 0
    NaN in X_test:  0
    Inf in X_train: 0
    X_train dtype:  {dtype('int64'): 74, dtype('float64'): 43}

  Saved model-ready files:
    01_initial_eda.png                                0.08 MB
    02_correlation_heatmap_top20.png                  0.27 MB
    03_feature_importance.png                         0.18 MB
    05_vif_analysis.png                               0.15 MB
    X_test.parquet                                   13.91 MB
    X_train.parquet                                  48.19 MB
    encoding_csvs                                     0.02 MB
    feature_correlatio

## 11 — Baseline Model (LightGBM)


In [30]:
print("=" * 90)
print("BASELINE MODEL — LightGBM Binary Classifier")
print("=" * 90)

if HAS_LGB:
    from sklearn.metrics import (classification_report, roc_auc_score, 
                                  precision_recall_curve, average_precision_score,
                                  confusion_matrix)
    
    params_baseline = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": 8,
        "min_child_samples": 50,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": (y_train==0).sum() / max((y_train==1).sum(), 1),
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": -1
    }
    
    train_ds = lgb.Dataset(X_train, label=y_train)
    val_ds = lgb.Dataset(X_test, label=y_test, reference=train_ds)
    
    print("\n  Training baseline model...")
    model_baseline = lgb.train(
        params_baseline, train_ds,
        num_boost_round=500,
        valid_sets=[val_ds],
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(100)]
    )
    
    # Predictions
    y_prob = model_baseline.predict(X_test)
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Metrics
    auc_roc = roc_auc_score(y_test, y_prob)
    avg_prec = average_precision_score(y_test, y_prob)
    
    print(f"\n  Baseline Results:")
    print(f"    AUC-ROC:            {auc_roc:.4f}")
    print(f"    Average Precision:  {avg_prec:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["Clean", "AML"], digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"  Confusion Matrix:")
    print(f"    {'':>15s} {'Pred Clean':>12s} {'Pred AML':>12s}")
    print(f"    {'Actual Clean':<15s} {cm[0,0]:>12,} {cm[0,1]:>12,}")
    print(f"    {'Actual AML':<15s} {cm[1,0]:>12,} {cm[1,1]:>12,}")
    
    # Per-typology recall
    if "aml_typology" in df.columns:
        print(f"\n  Per-Typology Recall (on test set):")
        test_df = df.iloc[X_test.index].copy()
        test_df["_pred_prob"] = y_prob
        test_df["_pred"] = y_pred
        
        typ_col = "aml_typology"
        all_typs = set()
        for t in test_df[typ_col].dropna():
            for part in str(t).split("; "):
                if part.strip(): all_typs.add(part.strip())
        
        print(f"    {'Typology':<40s} {'Total':>7s} {'Caught':>7s} {'Recall':>7s} {'Avg Prob':>9s}")
        print(f"    {'─'*75}")
        for typ in sorted(all_typs):
            mask = test_df[typ_col].astype(str).str.contains(typ, na=False)
            cnt = mask.sum()
            if cnt == 0: continue
            caught = test_df.loc[mask, "_pred"].sum()
            recall = caught / cnt * 100
            avg_prob = test_df.loc[mask, "_pred_prob"].mean()
            status = "✓" if recall > 80 else ("⚡" if recall > 50 else "⚠")
            print(f"    {typ:<40s} {cnt:>7,} {caught:>7,} {recall:>6.1f}% {avg_prob:>8.3f} {status}")
    
    # Plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # ROC curve
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, "b-", lw=2, label=f"AUC = {auc_roc:.4f}")
    axes[0].plot([0,1], [0,1], "k--", alpha=0.3)
    axes[0].set_title("ROC Curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
    axes[0].legend()
    
    # PR curve
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec, prec, "r-", lw=2, label=f"AP = {avg_prec:.4f}")
    axes[1].set_title("Precision-Recall Curve"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].legend()
    
    # Score distribution
    axes[2].hist(y_prob[y_test==0], bins=50, alpha=0.6, label="Clean", color="#2ecc71", density=True)
    axes[2].hist(y_prob[y_test==1], bins=50, alpha=0.6, label="AML", color="#e74c3c", density=True)
    axes[2].set_title("Score Distribution"); axes[2].set_xlabel("Predicted Probability")
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "04_baseline_model_results.png"), bbox_inches="tight")
    plt.show()
    print(f"\n  Saved: {OUTPUT_DIR}/04_baseline_model_results.png")
    
    # Save model
    model_baseline.save_model(os.path.join(OUTPUT_DIR, "baseline_lgb_model.txt"))
    print(f"  Saved: {OUTPUT_DIR}/baseline_lgb_model.txt")
else:
    print("  Skipped — LightGBM not available")

print(f"\n{'='*90}")
print("ML PREPARATION COMPLETE")
print(f"{'='*90}")
print(f"\n  Outputs in: {os.path.abspath(OUTPUT_DIR)}/")
print(f"  Next steps:")
print(f"    1. Review feature_correlations.csv for manual feature curation")
print(f"    2. Use X_train/X_test/y_train/y_test for hyperparameter tuning")
print(f"    3. Build Phase 2 typology classifier on is_aml=1 subset")



BASELINE MODEL — LightGBM Binary Classifier

  Training baseline model...
Training until validation scores don't improve for 30 rounds
[100]	valid_0's auc: 0.85995
[200]	valid_0's auc: 0.870108
[300]	valid_0's auc: 0.875299
[400]	valid_0's auc: 0.878564
[500]	valid_0's auc: 0.880771
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.880771

  Baseline Results:
    AUC-ROC:            0.8808
    Average Precision:  0.8436

  Classification Report:
              precision    recall  f1-score   support

       Clean     0.8479    0.8262    0.8369     40846
         AML     0.7368    0.7665    0.7513     25929

    accuracy                         0.8030     66775
   macro avg     0.7923    0.7963    0.7941     66775
weighted avg     0.8047    0.8030    0.8037     66775

  Confusion Matrix:
                      Pred Clean     Pred AML
    Actual Clean          33,746        7,100
    Actual AML             6,055       19,874

  Per-Typology Recall (on test set):
    Ty